# Fetch NMDC per-gene KO + EC annotations (issue #83)

Downloads BLAST-style per-gene annotation TSVs from `data.microbiomedata.org`,
parses them, and writes one Parquet per `data_object_type`.

| `data_object_type` | ~files | ~total | ~avg | Format |
|---|---|---|---|---|
| `Annotation KEGG Orthology` | 4,776 | 163 GB | 35 MB | no-header BLAST TSV (11 cols) |
| `Annotation Enzyme Commission` | 4,771 | 111 GB | 24 MB | no-header BLAST TSV (11 cols) |

**Disk footprint:** ~274 GB of raw downloads will be cached under
`OUT_DIR/raw_cache/`. Set `DOWNLOAD_CACHE_DIR` to override (e.g. point at a
scratch volume) or delete the dir to reclaim space after Parquet writes.

Each Parquet row carries `workflow_run_id` (= `was_generated_by` from
`data_object_set`) and `data_object_id` for downstream joins.

Same architectural pattern as `fetch_taxonomy_summaries.ipynb` — see
`LOAD_TAXONOMY_NOTES.md` for the on-pod Spark/auth pattern, NMDC quirks
(placeholder files, duplicate URLs, 404 gaps), and design rationale.

### Imports + config + logger

In [ ]:
import inspect
import io
import logging
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential

# get_spark_session is auto-imported by the BERDL kernel startup script.

OUT_DIR = Path(os.environ.get("KO_EC_OUT_DIR", "loaded_ko_ec"))
OUT_DIR.mkdir(exist_ok=True)

# Raw downloads are large (~274 GB total). Override DOWNLOAD_CACHE_DIR to put them
# on a different volume if needed.
CACHE_DIR = Path(os.environ.get("DOWNLOAD_CACHE_DIR", str(OUT_DIR / "raw_cache")))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

HTTP_WORKERS = int(os.environ.get("HTTP_WORKERS", "32"))
DRY_RUN      = False

LOG_DIR  = OUT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger("nmdc_lakehouse.ko_ec")
logger.setLevel(logging.INFO)
for _h in list(logger.handlers):
    logger.removeHandler(_h)
_fh = logging.FileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(_sh)
logger.propagate = False

print(f"OUT_DIR:   {OUT_DIR.resolve()}")
print(f"CACHE_DIR: {CACHE_DIR.resolve()}")
print(f"LOG_FILE:  {LOG_FILE.resolve()}")

### Spark Connect

In [ ]:
spark = get_spark_session(app_name="fetch_ko_ec_annotations")
print(f"Spark version: {spark.version}")

### Fetch the URL manifest from `data_object_set`

In [ ]:
TARGET_TYPES = [
    "Annotation KEGG Orthology",
    "Annotation Enzyme Commission",
]

type_list = ", ".join(f"'{t}'" for t in TARGET_TYPES)

manifest_sql = f"""
SELECT id, url, data_object_type, was_generated_by, file_size_bytes, md5_checksum
FROM nmdc_metadata.data_object_set
WHERE data_object_type IN ({type_list})
  AND url IS NOT NULL
  AND url LIKE 'https://data.microbiomedata.org/%'
ORDER BY data_object_type, id
"""

t0 = time.monotonic()
manifest = spark.sql(manifest_sql).toPandas()
logger.info(f"manifest fetched in {time.monotonic() - t0:.1f}s")

# Dedup: data_object_set sometimes has multiple ids pointing at the same URL.
n_before = len(manifest)
manifest = manifest.drop_duplicates(subset=["url", "data_object_type"]).reset_index(drop=True)
if len(manifest) != n_before:
    logger.info(f"deduped: {n_before:,} → {len(manifest):,} (dropped {n_before - len(manifest):,} duplicates)")

logger.info(f"{len(manifest):,} unique data objects to process")
# file_size_bytes may be string-typed from the Parquet schema — cast defensively.
size_gb = pd.to_numeric(manifest["file_size_bytes"], errors="coerce").fillna(0).sum() / 1024**3
logger.info(f"size estimate: {size_gb:.1f} GB total")
print(manifest.groupby("data_object_type").size().to_string())

### Save the manifest + offload the actual download to a terminal

The kernel was crashing during the in-notebook parallel HTTP loop on this
274 GB run. Workaround: save the manifest to CSV and run
`scripts/download_to_cache.py` from a terminal. That populates the same
`raw_cache/` the notebook uses, but isn't subject to kernel restarts.

In a JupyterLab terminal (or any shell on the pod):

```bash
cd /home/mamillerpa/nmdc-lakehouse
nohup python scripts/download_to_cache.py \
    --manifest notebooks/loaded_ko_ec/manifest.csv \
    --cache-dir notebooks/loaded_ko_ec/raw_cache \
    --workers 8 \
    > notebooks/loaded_ko_ec/download.log 2>&1 &

tail -f notebooks/loaded_ko_ec/download.log
```

Once the script finishes, come back and run the parse loop below — it sees
100% cache hits.

In [ ]:
manifest_csv = OUT_DIR / "manifest.csv"
manifest.to_csv(manifest_csv, index=False)
print(f"manifest written: {manifest_csv}  ({len(manifest):,} rows)")
print()
print("To run the standalone downloader, paste this into a terminal:")
print()
print(f"  cd {Path('.').resolve()}")
print(f"  nohup python scripts/download_to_cache.py \\")
print(f"      --manifest {manifest_csv.relative_to(Path('.').resolve())} \\")
print(f"      --cache-dir {CACHE_DIR.relative_to(Path('.').resolve())} \\")
print(f"      --workers 8 \\")
print(f"      > {(OUT_DIR / 'download.log').relative_to(Path('.').resolve())} 2>&1 &")
print()
print(f"  tail -f {(OUT_DIR / 'download.log').relative_to(Path('.').resolve())}")

### Format parsers — BLAST-style TSV (no header, 11 columns)

Both KO and EC files share the same schema. The third column carries the
annotation identifier (e.g. `KO:K00835` or `EC:2.6.1.66`); we keep the prefix
so the table is self-describing and unambiguous.

Placeholder detection is the same defensive pattern from the taxonomy loader —
we don't yet know if KO/EC use single-line "No results" placeholders, but the
`_is_placeholder` check is cheap and consistent.

In [ ]:
_PLACEHOLDER_PATTERNS = (
    "Nothing found in",
    "No KO Results for",
    "No EC Results for",
    "No KEGG Results for",
    "No Enzyme",
    "No annotations",
    "No Annotation",
)


def _is_placeholder(text: str) -> bool:
    stripped = text.strip()
    if not stripped:
        return True
    if "\n" in stripped:
        return False
    return any(stripped.startswith(p) for p in _PLACEHOLDER_PATTERNS)


# BLAST output schema, per issue #83 example:
#   gene_id  ncbi_taxid  KO:Kxxxxx  pct_identity  q_start  q_end  s_start  s_end  evalue  bitscore  aln_len
_BLAST_COLS = [
    "gene_id", "ncbi_taxid", "annotation_id",
    "pct_identity", "q_start", "q_end", "s_start", "s_end",
    "evalue", "bitscore", "aln_len",
]


def _parse_blast_tsv(text: str) -> pd.DataFrame | None:
    if _is_placeholder(text):
        return None
    df = pd.read_csv(
        io.StringIO(text),
        sep="\t",
        header=None,
        names=_BLAST_COLS,
        # Some files may not have all 11 cols on every line — be tolerant.
        on_bad_lines="warn",
        engine="python",
    )
    if df.empty:
        return None
    return df


# Per-type parser hooks (identical implementation today, but separated so a
# future divergence in format is easy to slot in).
_PARSERS = {
    "Annotation KEGG Orthology":     _parse_blast_tsv,
    "Annotation Enzyme Commission":  _parse_blast_tsv,
}

### Download + parse loop (parallel, with disk cache)

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "nmdc-lakehouse/fetch_ko_ec_annotations"})


@retry(
    retry=retry_if_exception_type(requests.RequestException),
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    reraise=True,
)
def fetch_text(url: str, timeout: int = 120) -> str:
    # Larger default timeout: KO/EC files average 24-35 MB.
    r = SESSION.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text


def _cache_path_for(url: str) -> Path:
    return CACHE_DIR / urlparse(url).path.lstrip("/")


def fetch_text_cached(url: str, timeout: int = 120) -> str:
    path = _cache_path_for(url)
    if path.exists():
        return path.read_text()
    text = fetch_text(url, timeout=timeout)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(f"{path.suffix}.{threading.get_ident()}.tmp")
    tmp.write_text(text)
    tmp.replace(path)
    return text


def _fetch_and_parse(row, parser):
    try:
        text = fetch_text_cached(row.url)
        df   = parser(text)
        if df is None:
            return None, None  # placeholder / empty — not an error
        df["workflow_run_id"] = row.was_generated_by
        df["data_object_id"] = row.id
        return df, None
    except Exception as exc:
        return None, (row.url, str(exc))


def _stabilize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """For each object-dtype column, try numeric coercion; fall back to nullable string."""
    for col in df.columns:
        if df[col].dtype != object:
            continue
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            df[col] = df[col].astype("string")
    return df


def load_type(type_name: str, rows: pd.DataFrame) -> pd.DataFrame | None:
    parser = _PARSERS[type_name]
    frames: list[pd.DataFrame] = []
    errors: list[tuple[str, str]] = []
    n_placeholders = 0
    n_in_manifest  = len(rows)
    if DRY_RUN:
        rows = rows.head(1)
        logger.info(f"  DRY_RUN: processing 1 of {n_in_manifest} files")

    total = len(rows)
    cache_hits = sum(1 for r in rows.itertuples() if _cache_path_for(r.url).exists())
    logger.info(f"  cache: {cache_hits:,}/{total:,} on disk; will fetch {total - cache_hits:,}")

    with ThreadPoolExecutor(max_workers=HTTP_WORKERS) as ex:
        futures = [ex.submit(_fetch_and_parse, row, parser) for row in rows.itertuples()]
        for i, fut in enumerate(as_completed(futures), 1):
            df, err = fut.result()
            if df is not None:
                frames.append(df)
            elif err is not None:
                errors.append(err)
            else:
                n_placeholders += 1
            if i % 200 == 0 or i == 1 or i == total:
                print(f"  {i}/{total}…", end="", flush=True)

    print()
    logger.info(
        f"  parsed: {len(frames):,} files with data; "
        f"{n_placeholders:,} placeholders; {len(errors):,} errors"
    )
    if errors:
        for url, msg in errors[:5]:
            logger.info(f"    error: {url}: {msg}")
        if len(errors) > 5:
            logger.info(f"    … and {len(errors) - 5} more")

    if not frames:
        return None
    combined = pd.concat(frames, ignore_index=True)
    combined = _stabilize_dtypes(combined)
    logger.info(f"  combined: {len(combined):,} rows × {len(combined.columns)} cols")
    logger.info(f"  dtypes:   {dict(combined.dtypes.astype(str))}")
    return combined

### Main: one Parquet per `data_object_type`

Sanity check at top fails fast (in <1s) if the kernel has stale code from a
previous edit — saves multi-minute runs that would have crashed at the end.

In [ ]:
# Sanity check: the in-kernel functions match the latest cell source
assert "_stabilize_dtypes" in inspect.getsource(load_type), \
    "Kernel has STALE load_type — re-run its defining cell."
assert "fetch_text_cached" in inspect.getsource(_fetch_and_parse), \
    "Kernel has STALE _fetch_and_parse — re-run its defining cell."
logger.info("sanity check passed: in-kernel parsers and loader are current.")


def _safe(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


summary: dict[str, int] = {}
t_run = time.monotonic()

for type_name, group in manifest.groupby("data_object_type"):
    out_path = OUT_DIR / f"{_safe(type_name)}.parquet"
    logger.info(f"\n{'─'*60}")
    logger.info(f"{type_name} ({len(group):,} files → {out_path.name})")

    combined = load_type(type_name, group)
    if combined is None:
        logger.info("  skipped (no data)")
        continue

    t_write = time.monotonic()
    combined.to_parquet(out_path, index=False)
    logger.info(f"  wrote {out_path} in {time.monotonic() - t_write:.1f}s ({out_path.stat().st_size / 1024**2:,.1f} MB)")
    summary[type_name] = len(combined)

elapsed = time.monotonic() - t_run
logger.info(f"\n{'='*60}")
logger.info(f"Done in {elapsed/60:.1f} min")
for t, n in summary.items():
    logger.info(f"  {t}: {n:,} rows")
logger.info(f"Full log: {LOG_FILE}")

### Spot-check

In [ ]:
for fname in ["annotation_kegg_orthology.parquet", "annotation_enzyme_commission.parquet"]:
    p = OUT_DIR / fname
    if not p.exists():
        print(f"-- {fname}: not written")
        continue
    df = pd.read_parquet(p)
    print(f"\n=== {fname}: {df.shape} ===")
    print(df.dtypes)
    print(df.head(3).to_string())
    print(f"distinct workflow runs: {df['workflow_run_id'].nunique():,}")
    print(f"distinct annotation_ids: {df['annotation_id'].nunique():,}")

## Next steps

Run `ingest_ko_ec_annotations.ipynb` to upload these Parquets to MinIO bronze
and register them as Delta tables under `nmdc_results` namespace.